In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


path_to_class_folders="/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat"


import os
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from torch.cuda.amp import autocast, GradScaler


ROOT_DIR = "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat"

SUBFOLDER = "In_focus"


SELECTED_CLASSES = [
    "Thalassiosira sp",
    "Diatom 4 (c. concavicornus)",
    "Diatom 3 (ditylum sp.)",
    "Diatom 2",
    "Diatom 1 (c. debilis)",
    "Copepod Nauplii",
    "Copepod",
    "Ciliate",
    "Ceratium muelleri (singular)",
    "Ceratium furca (singular)"
]

CLASS_PATHS = [
    os.path.join(ROOT_DIR, cls)
    for cls in SELECTED_CLASSES
]


import os
from PIL import Image
from torch.utils.data import Dataset

class PlanktonDataset(Dataset):

    def __init__(self,
                 class_paths,
                 subfolder="In_focus",
                 transform=None):

        self.transform = transform
        self.samples = []

        #################################################
        # Label Mapping
        #################################################

        self.class_names = [
            os.path.basename(path)
            for path in class_paths
        ]

        self.class_to_idx = {
            cls: idx
            for idx, cls in enumerate(self.class_names)
        }

        #################################################
        # Read Images
        #################################################

        valid_extensions = (".tif", ".tiff")

        for class_path in class_paths:

            class_name = os.path.basename(class_path)

            label = self.class_to_idx[class_name]

            image_folder = os.path.join(class_path,
                                        subfolder)

            if not os.path.exists(image_folder):

                print(f"Warning : {image_folder} not found")

                continue

            for image in sorted(os.listdir(image_folder)):

                if image.lower().endswith(valid_extensions):

                    image_path = os.path.join(image_folder,
                                              image)

                    self.samples.append(
                        (image_path,
                         label)
                    )

        print("="*50)
        print("Selected Classes :", len(self.class_names))
        print("Total Images     :", len(self.samples))
        print("="*50)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])


from torchvision import transforms

IMG_SIZE = 128
# Validation/Test transformations
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


full_dataset = PlanktonDataset(
    class_paths=CLASS_PATHS,
    subfolder="In_focus",
    transform=None
)

print("Total images:", len(full_dataset))
print("Classes:", full_dataset.class_names)
print("Number of classes:", len(full_dataset.class_names))



from sklearn.model_selection import train_test_split

# --------------------------------------------------
# Extract labels directly from the full dataset
# Keep them as normal Python integers
# --------------------------------------------------

dataset_labels = [
    int(label)
    for _, label in full_dataset.samples
]

# All dataset indices
indices = list(range(len(full_dataset)))

print("Number of images :", len(indices))
print("Number of labels :", len(dataset_labels))

# --------------------------------------------------
# 70% Train / 30% Temporary
# --------------------------------------------------

train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=dataset_labels
)

print("Train :", len(train_indices))
print("Temp  :", len(temp_indices))


from sklearn.model_selection import train_test_split

# --------------------------------------------------
# Extract labels directly from the full dataset
# Keep them as normal Python integers
# --------------------------------------------------

dataset_labels = [
    int(label)
    for _, label in full_dataset.samples
]

# All dataset indices
indices = list(range(len(full_dataset)))

print("Number of images :", len(indices))
print("Number of labels :", len(dataset_labels))

# --------------------------------------------------
# 70% Train / 30% Temporary
# --------------------------------------------------

train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=dataset_labels
)

print("Train :", len(train_indices))
print("Temp  :", len(temp_indices))


# Get labels corresponding only to temp_indices
temp_labels = [
    dataset_labels[i]
    for i in temp_indices
]

# Split the remaining 30% into:
# 15% validation
# 15% test
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("\nFinal Split")
print("=" * 40)
print("Train      :", len(train_indices))
print("Validation :", len(val_indices))
print("Test       :", len(test_indices))


dataset = PlanktonDataset(
    class_paths=CLASS_PATHS,
    subfolder="In_focus",
    transform=train_transform
)


from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

Selected Classes : 10
Total Images     : 7000
Total images: 7000
Classes: ['Thalassiosira sp', 'Diatom 4 (c. concavicornus)', 'Diatom 3 (ditylum sp.)', 'Diatom 2', 'Diatom 1 (c. debilis)', 'Copepod Nauplii', 'Copepod', 'Ciliate', 'Ceratium muelleri (singular)', 'Ceratium furca (singular)']
Number of classes: 10
Number of images : 7000
Number of labels : 7000
Train : 4900
Temp  : 2100
Number of images : 7000
Number of labels : 7000
Train : 4900
Temp  : 2100

Final Split
Train      : 4900
Validation : 1050
Test       : 1050
Selected Classes : 10
Total Images     : 7000
torch.Size([8, 3, 128, 128])
torch.Size([8])


In [2]:
import torch
import torch.nn as nn


class DIBB(nn.Module):
    """
    Depthwise Inverse Bottleneck Block
    """

    def __init__(
        self,
        in_channels,
        out_channels,
        expansion_ratio=2,
        stride=1
    ):
        super().__init__()

        self.stride = stride
        self.in_channels = in_channels
        self.out_channels = out_channels

        ####################################################
        # Hidden Channels
        ####################################################

        hidden_channels = int(
            in_channels * expansion_ratio
        )

        ####################################################
        # Expansion Layer
        ####################################################

        self.expand = nn.Sequential(

            nn.Conv2d(
                in_channels,
                hidden_channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(hidden_channels),

            nn.ReLU6(inplace=True)

        )

        ####################################################
        # Depthwise Convolution
        ####################################################

        self.depthwise = nn.Sequential(

            nn.Conv2d(
                hidden_channels,
                hidden_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=hidden_channels,
                bias=False
            ),

            nn.BatchNorm2d(hidden_channels),

            nn.ReLU6(inplace=True)

        )

        ####################################################
        # Projection Layer
        ####################################################

        self.project = nn.Sequential(

            nn.Conv2d(
                hidden_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels)

        )

        ####################################################
        # Residual Connection
        ####################################################

        self.use_residual = (
            stride == 1 and
            in_channels == out_channels
        )

    def forward(self, x):

        identity = x

        out = self.expand(x)

        out = self.depthwise(out)

        out = self.project(out)

        if self.use_residual:
            out = out + identity

        return out

In [3]:
block = DIBB(
    in_channels=32,
    out_channels=64,
    expansion_ratio=2,
    stride=2
)

x = torch.randn(
    1,
    32,
    128,
    128
)

y = block(x)

print(x.shape)
print(y.shape)

torch.Size([1, 32, 128, 128])
torch.Size([1, 64, 64, 64])


In [4]:
total_params = sum(
    p.numel()
    for p in block.parameters()
)

trainable_params = sum(
    p.numel()
    for p in block.parameters()
    if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 7,104
Trainable Parameters : 7,104


In [5]:
import torch
import torch.nn as nn


class ChannelAttention(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16
    ):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(

            nn.Conv2d(
                channels,
                channels // reduction,
                kernel_size=1,
                bias=False
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                channels // reduction,
                channels,
                kernel_size=1,
                bias=False
            )

        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = self.mlp(
            self.avg_pool(x)
        )

        max_out = self.mlp(
            self.max_pool(x)
        )

        attention = self.sigmoid(
            avg_out + max_out
        )

        return x * attention

In [6]:
x = torch.randn(
    1,
    64,
    128,
    128
)

ca = ChannelAttention(
    channels=64,
    reduction=16
)

y = ca(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([1, 64, 128, 128])
Output: torch.Size([1, 64, 128, 128])


In [7]:
import torch
import torch.nn as nn


class SpatialAttention(nn.Module):

    def __init__(self, kernel_size=7):

        super().__init__()

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            in_channels=2,
            out_channels=1,
            kernel_size=kernel_size,
            padding=padding,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        ####################################################
        # Average Pooling along Channel Dimension
        ####################################################

        avg_pool = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        ####################################################
        # Max Pooling along Channel Dimension
        ####################################################

        max_pool, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        ####################################################
        # Concatenate
        ####################################################

        pooled = torch.cat(
            [avg_pool, max_pool],
            dim=1
        )

        ####################################################
        # Spatial Attention
        ####################################################

        attention = self.sigmoid(
            self.conv(pooled)
        )

        ####################################################
        # Multiply
        ####################################################

        return x * attention

In [8]:
x = torch.randn(
    1,
    64,
    128,
    128
)

sa = SpatialAttention()

y = sa(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([1, 64, 128, 128])
Output: torch.Size([1, 64, 128, 128])


In [9]:
import torch.nn as nn


class DualChannelAttention(nn.Module):
    """
    Dual Channel Attention (CBAM)

    Channel Attention
            ↓
    Spatial Attention
    """

    def __init__(
        self,
        channels,
        reduction=16,
        kernel_size=7
    ):
        super().__init__()

        self.channel_attention = ChannelAttention(
            channels=channels,
            reduction=reduction
        )

        self.spatial_attention = SpatialAttention(
            kernel_size=kernel_size
        )

    def forward(self, x):

        x = self.channel_attention(x)

        x = self.spatial_attention(x)

        return x

In [10]:
dca = DualChannelAttention(64)

total_params = sum(
    p.numel()
    for p in dca.parameters()
)

print(f"Total Parameters : {total_params:,}")

Total Parameters : 610


In [11]:
class DIBBStage(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        num_blocks,
        expansion_ratio=2,
        stride=1
    ):
        super().__init__()

        layers = []

        ###################################################
        # First Block
        ###################################################

        layers.append(
            DIBB(
                in_channels=in_channels,
                out_channels=out_channels,
                expansion_ratio=expansion_ratio,
                stride=stride
            )
        )

        ###################################################
        # Remaining Blocks
        ###################################################

        for _ in range(num_blocks - 1):

            layers.append(

                DIBB(
                    in_channels=out_channels,
                    out_channels=out_channels,
                    expansion_ratio=expansion_ratio,
                    stride=1
                )

            )

        self.stage = nn.Sequential(*layers)

    def forward(self, x):

        return self.stage(x)

In [12]:
stage1 = DIBBStage(
    in_channels=32,
    out_channels=32,
    num_blocks=2,
    stride=1
)

x = torch.randn(1,32,64,64)

y = stage1(x)

print(y.shape)

torch.Size([1, 32, 64, 64])


In [13]:
class Stem(nn.Module):

    def __init__(self):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(
                3,
                32,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(32),

            nn.ReLU6(inplace=True)

        )

    def forward(self, x):

        return self.stem(x)

In [14]:
stem = Stem()

x = torch.randn(1,3,128,128)

y = stem(x)

print(y.shape)

torch.Size([1, 32, 64, 64])


In [15]:
class DIBBBackbone(nn.Module):

    def __init__(self):

        super().__init__()

        ##################################################
        # Stem
        ##################################################

        self.stem = Stem()

        ##################################################
        # Stage 1
        ##################################################

        self.stage1 = DIBBStage(
            32,
            32,
            num_blocks=2,
            stride=1
        )

        self.dca1 = DualChannelAttention(32)

        ##################################################
        # Stage 2
        ##################################################

        self.stage2 = DIBBStage(
            32,
            64,
            num_blocks=3,
            stride=2
        )

        self.dca2 = DualChannelAttention(64)

        ##################################################
        # Stage 3
        ##################################################

        self.stage3 = DIBBStage(
            64,
            128,
            num_blocks=2,
            stride=2
        )

        self.dca3 = DualChannelAttention(128)

        ##################################################
        # Stage 4
        ##################################################

        self.stage4 = DIBBStage(
            128,
            256,
            num_blocks=1,
            stride=2
        )

    def forward(self, x):

        x = self.stem(x)

        f1 = self.dca1(
            self.stage1(x)
        )

        f2 = self.dca2(
            self.stage2(f1)
        )

        f3 = self.dca3(
            self.stage3(f2)
        )

        f4 = self.stage4(f3)

        return f1, f2, f3, f4

In [16]:
model = DIBBBackbone()

x = torch.randn(
    1,
    3,
    128,
    128
)

f1, f2, f3, f4 = model(x)

print("F1 :", f1.shape)
print("F2 :", f2.shape)
print("F3 :", f3.shape)
print("F4 :", f4.shape)

F1 : torch.Size([1, 32, 64, 64])
F2 : torch.Size([1, 64, 32, 32])
F3 : torch.Size([1, 128, 16, 16])
F4 : torch.Size([1, 256, 8, 8])


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FeaturePyramid(nn.Module):

    def __init__(self, out_channels=128):
        super().__init__()

        ##################################################
        # Lateral Convolutions
        ##################################################

        self.lat1 = nn.Conv2d(32, out_channels, kernel_size=1)
        self.lat2 = nn.Conv2d(64, out_channels, kernel_size=1)
        self.lat3 = nn.Conv2d(128, out_channels, kernel_size=1)
        self.lat4 = nn.Conv2d(256, out_channels, kernel_size=1)

        ##################################################
        # Smoothing Convolutions
        ##################################################

        self.smooth1 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth4 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, f1, f2, f3, f4):
        ###############################################
        # Lateral Features
        ###############################################

        p4 = self.lat4(f4)
        p3 = self.lat3(f3)
        p2 = self.lat2(f2)
        p1 = self.lat1(f1)

        ###############################################
        # Top-Down Pathway
        ###############################################

        p3 = p3 + F.interpolate(p4, scale_factor=2, mode="nearest")
        p2 = p2 + F.interpolate(p3, scale_factor=2, mode="nearest")
        p1 = p1 + F.interpolate(p2, scale_factor=2, mode="nearest")

        ###############################################
        # Smoothing
        ###############################################

        p4 = self.smooth4(p4)
        p3 = self.smooth3(p3)
        p2 = self.smooth2(p2)
        p1 = self.smooth1(p1)

        return p1, p2, p3, p4


fpn = FeaturePyramid()

f1 = torch.randn(1, 32, 64, 64)
f2 = torch.randn(1, 64, 32, 32)
f3 = torch.randn(1, 128, 16, 16)
f4 = torch.randn(1, 256, 8, 8)

p1, p2, p3, p4 = fpn(f1, f2, f3, f4)

print(p1.shape)
print(p2.shape)
print(p3.shape)
print(p4.shape)

torch.Size([1, 128, 64, 64])
torch.Size([1, 128, 32, 32])
torch.Size([1, 128, 16, 16])
torch.Size([1, 128, 8, 8])


In [24]:
class MainClassifier(nn.Module):

    def __init__(self,
                 in_channels=256,
                 num_classes=10):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(
            in_channels,
            num_classes
        )

    def forward(self, x):

        x = self.pool(x)

        x = torch.flatten(x, 1)

        logits = self.classifier(x)

        return logits

In [25]:
import torch
import torch.nn as nn


class PyramidClassifier(nn.Module):

    def __init__(self,
                 channels=128,
                 num_classes=10):

        super().__init__()

        ####################################################
        # Feature Refinement Block
        ####################################################

        self.refine1 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.refine2 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.refine3 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.refine4 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        ####################################################
        # Global Average Pooling
        ####################################################

        self.pool = nn.AdaptiveAvgPool2d(1)

        ####################################################
        # Final Classifier
        ####################################################

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(channels * 4, num_classes)

    def forward(self, p1, p2, p3, p4):

        p1 = self.refine1(p1)
        p2 = self.refine2(p2)
        p3 = self.refine3(p3)
        p4 = self.refine4(p4)

        p1 = self.pool(p1).flatten(1)
        p2 = self.pool(p2).flatten(1)
        p3 = self.pool(p3).flatten(1)
        p4 = self.pool(p4).flatten(1)

        features = torch.cat(
            [p1, p2, p3, p4],
            dim=1
        )

        features = self.dropout(features)

        logits = self.fc(features)

        return logits

In [28]:
class DIBBNet(nn.Module):

    def __init__(self,
                 num_classes=10):

        super().__init__()

        self.backbone = DIBBBackbone()

        self.fpn = FeaturePyramid(
            out_channels=128
        )

        self.main_head = MainClassifier(
            in_channels=256,
            num_classes=num_classes
        )

        self.fpn_head = PyramidClassifier(
            channels=128,
            num_classes=num_classes
        )

    def forward(self, x):

        ##################################################
        # Backbone
        ##################################################

        f1, f2, f3, f4 = self.backbone(x)

        ##################################################
        # Main Output
        ##################################################

        main_logits = self.main_head(f4)

        ##################################################
        # Pyramid
        ##################################################

        p1, p2, p3, p4 = self.fpn(
            f1,
            f2,
            f3,
            f4
        )

        ##################################################
        # Pyramid Output
        ##################################################

        pyramid_logits = self.fpn_head(
            p1,
            p2,
            p3,
            p4
        )

        return main_logits, pyramid_logits

In [29]:
model = DIBBNet(
    num_classes=10
)

x = torch.randn(
    2,
    3,
    128,
    128
)

main_logits, pyramid_logits = model(x)

print(main_logits.shape)
print(pyramid_logits.shape)

torch.Size([2, 10])
torch.Size([2, 10])


In [30]:
criterion = nn.CrossEntropyLoss()

lambda_fpn = 0.5

In [31]:
main_logits, pyramid_logits = model(images)

loss_main = criterion(
    main_logits,
    labels
)

loss_fpn = criterion(
    pyramid_logits,
    labels
)

loss = loss_main + lambda_fpn * loss_fpn

In [32]:
prediction = torch.argmax(main_logits, dim=1)

In [33]:
def validate(model,
             val_loader,
             criterion,
             device,
             lambda_fpn=0.5):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            main_logits, pyramid_logits = model(images)

            loss_main = criterion(main_logits, labels)
            loss_fpn = criterion(pyramid_logits, labels)

            loss = loss_main + lambda_fpn * loss_fpn

            running_loss += loss.item()

            predictions = torch.argmax(main_logits, dim=1)

            total += labels.size(0)

            correct += (predictions == labels).sum().item()

    validation_loss = running_loss / len(val_loader)

    validation_accuracy = 100 * correct / total

    return validation_loss, validation_accuracy

In [35]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [37]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DIBBNet(num_classes=10)

model = model.to(device)

In [42]:
from torch.utils.data import Subset

####################################################
# Base Datasets
####################################################

train_base = PlanktonDataset(
    class_paths=CLASS_PATHS,
    subfolder="In_focus",
    transform=train_transform
)

val_base = PlanktonDataset(
    class_paths=CLASS_PATHS,
    subfolder="In_focus",
    transform=test_transform
)

test_base = PlanktonDataset(
    class_paths=CLASS_PATHS,
    subfolder="In_focus",
    transform=test_transform
)

####################################################
# Subsets
####################################################

train_dataset = Subset(train_base, train_indices)

val_dataset = Subset(val_base, val_indices)

test_dataset = Subset(test_base, test_indices)

Selected Classes : 10
Total Images     : 7000
Selected Classes : 10
Total Images     : 7000
Selected Classes : 10
Total Images     : 7000


In [44]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [45]:
EPOCHS = 30

criterion = nn.CrossEntropyLoss()

lambda_fpn = 0.5

best_val_acc = 0

for epoch in range(EPOCHS):

    ####################################################
    # Training
    ####################################################

    model.train()

    running_loss = 0.0

    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        main_logits, pyramid_logits = model(images)

        loss_main = criterion(main_logits, labels)

        loss_fpn = criterion(pyramid_logits, labels)

        loss = loss_main + lambda_fpn * loss_fpn

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = torch.argmax(main_logits, dim=1)

        train_total += labels.size(0)

        train_correct += (predictions == labels).sum().item()

    train_loss = running_loss / len(train_loader)

    train_accuracy = 100 * train_correct / train_total

    ####################################################
    # Validation
    ####################################################

    val_loss, val_accuracy = validate(
        model,
        val_loader,
        criterion,
        device,
        lambda_fpn
    )

    ####################################################
    # Save Best Model
    ####################################################

    if val_accuracy > best_val_acc:

        best_val_acc = val_accuracy

        torch.save(
            model.state_dict(),
            "best_dibb_fpn_model.pth"
        )

    ####################################################
    # Print Results
    ####################################################

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Acc : {val_accuracy:.2f}%")
    print("-"*60)

Epoch [1/30]
Train Loss     : 3.4583
Train Accuracy : 9.08%
Validation Loss: 3.4610
Validation Acc : 10.19%
------------------------------------------------------------
Epoch [2/30]
Train Loss     : 3.4589
Train Accuracy : 9.08%
Validation Loss: 3.4592
Validation Acc : 9.90%
------------------------------------------------------------
Epoch [3/30]
Train Loss     : 3.4612
Train Accuracy : 9.24%
Validation Loss: 3.4617
Validation Acc : 9.62%
------------------------------------------------------------
Epoch [4/30]
Train Loss     : 3.4597
Train Accuracy : 9.47%
Validation Loss: 3.4594
Validation Acc : 9.71%
------------------------------------------------------------
Epoch [5/30]
Train Loss     : 3.4587
Train Accuracy : 9.53%
Validation Loss: 3.4614
Validation Acc : 9.43%
------------------------------------------------------------
Epoch [6/30]
Train Loss     : 3.4587
Train Accuracy : 9.20%
Validation Loss: 3.4615
Validation Acc : 10.67%
---------------------------------------------------

Exception in thread Thread-28 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

KeyboardInterrupt: 

In [39]:
print(next(model.parameters()).device)

cpu
